In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()

src_path = repo_root / "../"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from doc_parsers.pdfParser import pdfParser
from doc_parsers.mdParser import md_parser


In [ ]:
BUNDLE_DIR = Path("../examples")  # where the bundle PDFs are
PDFS = [
    BUNDLE_DIR / "bundle_SOP_AFW_P101A.pdf",
    BUNDLE_DIR / "bundle_CR_2026_00123.pdf",
    BUNDLE_DIR / "bundle_WO_2026_04567.pdf",
    BUNDLE_DIR / "bundle_ECA_2026_0007.pdf",
]

DEST_ROOT = Path("parsed_docs")
DEST_ROOT.mkdir(parents=True, exist_ok=True)

PDFS, DEST_ROOT

In [ ]:
# Choose one PDF
pdf_path = PDFS[0]

# Since pdfParser wants (home_folder, red_filepath), set home_folder and make pdf relative
home_folder = str(BUNDLE_DIR)
red_filepath = pdf_path.relative_to(BUNDLE_DIR).as_posix()

doc_index = pdfParser(
    home_folder=home_folder,
    red_filepath=red_filepath,
    destination_folder=str(DEST_ROOT),
    text2markdown="marker",
    tableParser="marker",          # or "pdfplumber" :contentReference[oaicite:16]{index=16}
    classification="internal",
    ingest_id="demo_ingest_001",
    source_path=red_filepath,      # stored in document_index :contentReference[oaicite:17]{index=17}
)

structured = md_parser(
    document_index=doc_index,
    destination_folder=None,       # defaults to base folder above text_md_path :contentReference[oaicite:18]{index=18}
    mbse_entities=None,
    nureg_section_ids=None
)

doc_index.keys(), structured.keys()

In [ ]:
import json

doc_dirs = [p for p in DEST_ROOT.iterdir() if p.is_dir()]
doc_dirs

In [ ]:
# pick one doc folder
d = doc_dirs[0]
index_dir = d / "index"

# 3.2 Open a structured output and view section titles

structured_path = next(index_dir.glob("*_structured_output.json"))
chunks_path = next(index_dir.glob("*_chunks.jsonl"))

structured = json.loads(structured_path.read_text(encoding="utf-8"))

structured_path, chunks_path, structured.keys()

In [ ]:
# 3.3 Load chunks.jsonl and filter what gets embedded in Chroma
chunks = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines() if line.strip()]

# what chunk types do we have?
from collections import Counter
Counter([c["type"] for c in chunks])

# get only those intended for vector store
vector_chunks = [c for c in chunks if c.get("index_in_vector_store")]

len(vector_chunks), vector_chunks[0].keys()

In [ ]:
# Summarization
import os
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
os.environ["OLLAMA_MODEL"] = "mistral:latest"     # or your local model
os.environ["OLLAMA_NUM_CTX"] = "8192"

In [ ]:
from pathlib import Path

DEST_ROOT = Path("parsed_docs/fc4d8015d284")
chunks_files = sorted(DEST_ROOT.glob("index/*_chunks.jsonl"))
chunks_files

In [ ]:
#Run augmentation on one file
from ner.augment_chunks import augment_chunks_with_structured_summaries

stats = augment_chunks_with_structured_summaries(
    chunks_files[0],
    model=None,                  # uses env OLLAMA_MODEL
    overwrite=False,
    summarize_granularities=("section", "paragraph"),
    only_indexable=True,
)

stats